In [0]:
from pyspark.sql.functions import col, from_json, schema_of_json, get_json_object 
from pyspark.sql.types import _parse_datatype_string
from pyspark.sql.window import Window
from pyspark.sql.functions import * 
from pyspark.sql.types import *

In [0]:
df = spark.read.table("dev_stock.bronze.stocks_raw")
display(df)


In [0]:
# Add an event_type column to route your data
df = df.withColumn("event_type", get_json_object(col("trades"), "$.event_type"))

# --- 1. Get Ticker Schema ---
ticker = df.filter(col("event_type") == "ticker")
tschema = ticker.select(schema_of_json(col("trades")).alias("t_schema")).limit(1)
ddl_schema_str = tschema.first()["t_schema"] 
tschema = _parse_datatype_string(ddl_schema_str)

print("Ticker Schema---------------------")
print(tschema)

In [0]:
# Get Candle Schema 
candle = df.filter(col("event_type") == "candle")
cschema = candle.select(schema_of_json(col("trades")).alias("c_schema")).limit(1)
ddl_schema_str = cschema.first()["c_schema"] 
cschema = _parse_datatype_string(ddl_schema_str)

print("Candle Schema---------------------")
print(cschema)

In [0]:
# Get product Schema 
product = df.filter(col("event_type") == "product")
pschema = product.select(schema_of_json(col("trades")).alias("p_schema")).limit(1)
ddl_schema_str = pschema.first()["p_schema"] 
pschema = _parse_datatype_string(ddl_schema_str)

print("Product Schema---------------------")
print(pschema)

## Ticker

In [0]:
price_type = DecimalType(18, 2)   
crypto_type = DecimalType(12, 10)

ticker_df = ticker.withColumn("data", from_json(col("trades"), tschema)) \
                  .select("data.event_type", "data.ingested_at", "data.payload.*")

ticker_df = ticker_df.withColumn("single_trade", explode(col('trades')))\
                     .select("event_type", "ingested_at", "best_ask", "best_bid", "single_trade.*")

ticker_df = ticker_df.withColumn("trade_timestamp", col('time').cast('timestamp'))\
                     .withColumn("ingested_at", col('ingested_at').cast('timestamp'))\
                     .withColumn("price", col("price").cast(price_type))\
                     .withColumn("size", col("size").cast(crypto_type))\
                     .withColumn("best_ask", col("best_ask").cast('double'))\
                     .withColumn("best_bid", col("best_bid").cast('double'))

display(ticker_df)

## Product

In [0]:
product_df = product.withColumn("data", from_json(col('trades'), pschema))\
                    .select("data.event_type", "data.ingested_at", "data.payload.*") 

product_df = product_df.drop('alias', 'alias_to', 'about_description', 'auction_mode', 'base_cbrn', 'base_display_symbol', 'best_ask_price', 'best_bid_price', 'cancel_only', 'display_name', 'fcm_trading_session_details', 'high_24h', 'display_name_overwrite', 'icon_color', 'icon_url', 'product_cbrn', 'quote_cbrn', 'trading_disabled', 'view_only', 'is_alpha_testing', 'is_disabled', 'limit_only', 'low_24h', 'market_cap', 'mid_market_price')

product_df = product_df.withColumn('ingested_at', col('ingested_at').cast('timestamp'))\
                       .withColumn('new_at', col('new_at').cast('timestamp'))\
                       .withColumn('approximate_quote_24h_volume', col('approximate_quote_24h_volume').cast('double'))\
                       .withColumn('base_increment', col('base_increment').cast('double'))\
                       .withColumn('base_max_size', col('base_max_size').cast('long'))\
                       .withColumn('base_min_size', col('base_min_size').cast('double'))\
                       .withColumn('price', col('price').cast('double'))\
                       .withColumn('price_increment', col('price_increment').cast('double'))\
                       .withColumn('price_percentage_change_24h', col('price_percentage_change_24h').cast('double'))\
                       .withColumn('volume_24h', col('volume_24h').cast('double'))\
                       .withColumn('volume_percentage_change_24h', col('volume_percentage_change_24h').cast('double')) 

                           
                           
                     

display(product_df)

## Candle

In [0]:
candle_df = candle.withColumn("data", from_json(col('trades'), cschema))\
                  .select("data.event_type", "data.product_id", "data.ingested_at", "data.payload.*") 

candle_df = candle_df.withColumn('ingested_at', col('ingested_at').cast('timestamp'))\
                     .withColumn("start", from_unixtime(col("start")))\
                     .withColumn('close', col('close').cast('double'))\
                     .withColumn('high', col('high').cast('double'))\
                     .withColumn('low', col('low').cast('double'))\
                     .withColumn('open', col('open').cast('double'))\
                     .withColumn('volume', col('volume').cast('double'))\






display(candle_df)